# Cirrhosis Patient Survival Prediction

Predict, for each patient, the probability of each **Status** at the end of follow-up:

| Status | Meaning |
|---|---|
| **C** (censored) | alive at `N_Days`, no liver transplant |
| **CL** (censored, liver transplant) | alive at `N_Days` because of a liver transplant |
| **D** (death) | died by `N_Days` |

The metric is **multiclass log loss**, so the goal is well-calibrated probabilities, not just correct labels.

**Pipeline:** load data → clean → features (with an ablation) → stratified 5-fold CV for three gradient-boosting models plus baselines → test whether adding the original clinical dataset helps → fit blend weights on out-of-fold (OOF) predictions → write `submission.csv`.

Competition: [Mohirdev, *Sirroz bemorlari omon qolish prognozini aniqlash*](https://www.kaggle.com/competitions/multiclassificationtask).

In [1]:
import os
import glob
import warnings

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.5f}'.format)

SEED = 42
N_SPLITS = 5
CLASSES = ['C', 'CL', 'D']  # column order of every probability matrix below

print('lightgbm', lgb.__version__, '| xgboost', xgb.__version__, '| pandas', pd.__version__)

lightgbm 4.6.0 | xgboost 2.1.4 | pandas 2.3.3


## 1. Data

The competition data is **synthetic**: the organisers generated it with a deep-learning model trained on the real
[Cirrhosis Patient Survival Prediction](https://archive.ics.uci.edu/dataset/878/cirrhosis+patient+survival+prediction+dataset-1)
dataset (Mayo Clinic primary biliary cirrhosis trial, 418 patients). The organisers invite using that **original dataset** as extra
training data, which section 5 tests.

> Note: the competition's data page describes both C and CL as "alive at N_Days". That's a translation slip. In the original
> dataset, CL means the patient is alive *because of a liver transplant*.

Paths are resolved for both Kaggle (`/kaggle/input/...`) and a local `data/` folder. The original dataset is optional: if
`cirrhosis.csv` isn't found, section 5 is skipped.

In [2]:
def find_file(name, roots=('data', '/kaggle/input')):
    for root in roots:
        hits = glob.glob(os.path.join(root, '**', name), recursive=True)
        if hits:
            return sorted(hits)[0]
    return None

train_path, test_path = find_file('train.csv'), find_file('test.csv')
original_path = find_file('cirrhosis.csv')
if train_path is None or test_path is None:
    raise FileNotFoundError("Put train.csv and test.csv in 'data/' or attach the competition on Kaggle.")

train_raw = pd.read_csv(train_path)
test_raw = pd.read_csv(test_path)
original_raw = pd.read_csv(original_path) if original_path else None

print('train   :', train_raw.shape, train_path)
print('test    :', test_raw.shape, test_path)
print('original:', None if original_raw is None else original_raw.shape, original_path)

train   : (15000, 20) data/train.csv
test    : (10000, 19) data/test.csv
original: (418, 20) data/cirrhosis.csv


### Cleaning

* **Excluded label.** `train.csv` contains Status values outside C/CL/D. They aren't a competition class, so those rows are dropped.
* **Synthetic artefacts.** Some rows have impossible values, such as ages under 18 or `N_Days` equal to `Age`. These come from the
  generator. The test set has them too, so they're left in place; tree models are robust to them.
* **Missingness is informative.** About 43% of rows have no `Drug` and no lab panel, which mirrors the 106 non-trial patients in the
  original study. Missing values are passed to the boosters as-is rather than imputed, so that signal is preserved.

In [3]:
excluded = ~train_raw['Status'].isin(CLASSES)
print(f'Dropping {excluded.sum()} row(s) with an excluded Status label:')
display(train_raw.loc[excluded, ['id', 'Status', 'Ascites', 'Hepatomegaly', 'Spiders', 'Edema']])
train_raw = train_raw.loc[~excluded].reset_index(drop=True)

age_years = lambda df: df['Age'] / 365.25
print('Age < 18 years      -> train:', (age_years(train_raw) < 18).sum(), '| test:', (age_years(test_raw) < 18).sum())
print('N_Days equal to Age -> train:', (train_raw.N_Days == train_raw.Age).sum(), '| test:', (test_raw.N_Days == test_raw.Age).sum())
print('Rows missing Drug   -> train: {:.1%} | test: {:.1%}'.format(train_raw.Drug.isna().mean(), test_raw.Drug.isna().mean()))
print()
print(train_raw['Status'].value_counts(normalize=True).rename('class share'))

Dropping 1 row(s) with an excluded Status label:


,id,Status,Ascites,Hepatomegaly,Spiders,Edema
13789,13789,Y,Y,Y,Y,Y


Age < 18 years      -> train: 12 | test: 7
N_Days equal to Age -> train: 26 | test: 20
Rows missing Drug   -> train: 43.4% | test: 44.0%

Status
C    0.67024
D    0.30435
CL   0.02540
Name: class share, dtype: float64


## 2. Features

Engineered features are added in groups, and each group is kept only if it passes the ablation in section 4.

| Group | Features | Rationale |
|---|---|---|
| `base` | the 18 raw columns, `Age` in years | |
| `mayo` | **Mayo risk score** | The published Mayo Clinic survival model for primary biliary cirrhosis, developed on this same patient population:<br>`R = 0.039·age + 0.871·ln(bilirubin) − 2.53·ln(albumin) + 2.38·ln(prothrombin) + 0.859·edema`, with edema scored N=0, S=0.5, Y=1. Trees approximate additive combinations of features poorly, so handing them the clinical one directly can help. |
| `ratios` | bilirubin/albumin, SGOT/Alk_Phos, stage×bilirubin, prothrombin×bilirubin, platelets×prothrombin | interactions between liver-function markers |
| `adhoc` | two hand-weighted "liver scores" from earlier experiments | no published source; kept only if the data supports them |

Log transforms of single columns are deliberately **not** added. Tree splits don't change under monotone transforms, so they would be
redundant.

In [4]:
CAT_COLS = ['Drug', 'Sex', 'Ascites', 'Hepatomegaly', 'Spiders', 'Edema']
EDEMA_SCORE = {'N': 0.0, 'S': 0.5, 'Y': 1.0}
FEATURE_GROUPS = {
    'base': ['N_Days', 'Age', 'Bilirubin', 'Cholesterol', 'Albumin', 'Copper', 'Alk_Phos', 'SGOT',
             'Tryglicerides', 'Platelets', 'Prothrombin', 'Stage'] + CAT_COLS,
    'mayo': ['Mayo_risk'],
    'ratios': ['Bili_Albumin_ratio', 'SGOT_AlkPhos_ratio', 'Stage_Bili', 'Proth_Bili', 'Platelets_Prothrombin'],
    'adhoc': ['Liver_Score_1', 'Liver_Score_2'],
}


def engineer_features(df):
    df = df.copy()
    df['Age'] = df['Age'] / 365.25
    safe_log = lambda s: np.log(s.clip(lower=1e-3))

    df['Mayo_risk'] = (0.039 * df['Age'] + 0.871 * safe_log(df['Bilirubin']) - 2.53 * safe_log(df['Albumin'])
                       + 2.38 * safe_log(df['Prothrombin']) + 0.859 * df['Edema'].map(EDEMA_SCORE))

    df['Bili_Albumin_ratio'] = df['Bilirubin'] / (df['Albumin'] + 1e-6)
    df['SGOT_AlkPhos_ratio'] = df['SGOT'] / (df['Alk_Phos'] + 1e-6)
    df['Stage_Bili'] = df['Stage'] * df['Bilirubin']
    df['Proth_Bili'] = df['Prothrombin'] * df['Bilirubin']
    df['Platelets_Prothrombin'] = df['Platelets'] * df['Prothrombin']

    df['Liver_Score_1'] = 0.5 * df['Bilirubin'] + 0.003 * df['Cholesterol'] - 0.5 * df['Albumin']
    df['Liver_Score_2'] = np.log1p(df['Copper']) + np.log1p(df['Alk_Phos']) + np.log1p(df['SGOT'])
    return df


train = engineer_features(train_raw).assign(is_original=0)
test = engineer_features(test_raw).assign(is_original=0)
y = train['Status'].map({c: i for i, c in enumerate(CLASSES)}).to_numpy()

if original_raw is not None:
    original = engineer_features(original_raw[original_raw['Status'].isin(CLASSES)]).assign(is_original=1)
    y_original = original['Status'].map({c: i for i, c in enumerate(CLASSES)}).to_numpy()
else:
    original, y_original = None, None

# One shared category vocabulary, so codes line up between train, test and original rows.
for col in CAT_COLS:
    frames = [train, test] + ([original] if original is not None else [])
    vocab = sorted(set().union(*[f[col].dropna().unique() for f in frames]))
    for f in frames:
        f[col] = pd.Categorical(f[col], categories=vocab)

print('Mayo risk score by Status (median):')
print(train.groupby('Status')['Mayo_risk'].median())

Mayo risk score by Status (median):
Status
C    4.19970
CL   4.71096
D    5.58151
Name: Mayo_risk, dtype: float64


## 3. Validation setup

* **Stratified 5-fold CV** over competition rows, with the same folds for every model, so scores can be compared fold by fold.
* **Early stopping uses an inner 10% split of each training fold**, never the validation fold, so the OOF scores stay honest.
* **Categoricals are passed natively** to each booster: the pandas `category` dtype for LightGBM and XGBoost, `cat_features` for
  CatBoost. Nothing is scaled, since trees don't need it.
* **No class weighting.** Log loss rewards calibrated probabilities. Up-weighting the rare CL class would push its predicted
  probability above its true frequency and make log loss worse.
* Test predictions are averaged over the 5 fold models.

In [5]:
folds = list(StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED).split(train, y))


def fit_predict(model_name, X_tr, y_tr, X_pred_list, class_weight=None):
    X_fit, X_es, y_fit, y_es = train_test_split(X_tr, y_tr, test_size=0.1, stratify=y_tr, random_state=SEED)

    if model_name == 'lightgbm':
        model = lgb.LGBMClassifier(n_estimators=5000, learning_rate=0.03, num_leaves=31, min_child_samples=30,
                                   subsample=0.8, subsample_freq=1, colsample_bytree=0.6, reg_lambda=1.0,
                                   class_weight=class_weight, random_state=SEED, verbose=-1)
        model.fit(X_fit, y_fit, eval_set=[(X_es, y_es)], callbacks=[lgb.early_stopping(200, verbose=False)])
        return [model.predict_proba(X) for X in X_pred_list]

    if model_name == 'xgboost':
        model = xgb.XGBClassifier(n_estimators=5000, learning_rate=0.03, max_depth=5, min_child_weight=3,
                                  subsample=0.8, colsample_bytree=0.6, reg_lambda=1.0, tree_method='hist',
                                  enable_categorical=True, eval_metric='mlogloss', early_stopping_rounds=200,
                                  random_state=SEED)
        model.fit(X_fit, y_fit, eval_set=[(X_es, y_es)], verbose=False)
        return [model.predict_proba(X) for X in X_pred_list]

    if model_name == 'catboost':
        cats = [c for c in CAT_COLS if c in X_tr.columns]
        as_str = lambda X: X.astype({c: str for c in cats})  # CatBoost wants strings; NaN becomes 'nan'
        model = CatBoostClassifier(iterations=5000, learning_rate=0.05, depth=6, loss_function='MultiClass',
                                   early_stopping_rounds=200, random_seed=SEED, verbose=0, allow_writing_files=False)
        model.fit(as_str(X_fit), y_fit, cat_features=cats, eval_set=(as_str(X_es), y_es))
        return [model.predict_proba(as_str(X)) for X in X_pred_list]

    raise ValueError(model_name)


def run_cv(model_name, features, use_original=False, **fit_kwargs):
    if use_original:
        features = features + ['is_original']
    oof = np.zeros((len(train), 3))
    test_pred = np.zeros((len(test), 3))
    for tr_idx, va_idx in folds:
        X_tr, y_tr = train.iloc[tr_idx][features], y[tr_idx]
        if use_original:
            # Original rows only ever go on the training side; validation is competition rows only.
            X_tr = pd.concat([X_tr, original[features]], ignore_index=True)
            y_tr = np.concatenate([y_tr, y_original])
        va_pred, te_pred = fit_predict(model_name, X_tr, y_tr, [train.iloc[va_idx][features], test[features]],
                                       **fit_kwargs)
        oof[va_idx] = va_pred
        test_pred += te_pred / N_SPLITS
    return oof, test_pred


def fold_losses(oof):
    return np.array([log_loss(y[va], oof[va], labels=[0, 1, 2]) for _, va in folds])


def summarise(name, oof):
    f = fold_losses(oof)
    return {'model': name, 'OOF log loss': log_loss(y, oof), '± fold std': f.std()}

### Baselines

Two reference points for every score that follows:

* **Class frequencies**: predict the training class shares for every row. This is the "no model" floor.
* **Logistic regression** on the base features, with median imputation plus missing-value indicators, scaling and one-hot
  categoricals. A well-tuned linear model is the bar the boosters have to clear.

In [6]:
results = []

prior_oof = np.zeros((len(train), 3))
for tr_idx, va_idx in folds:
    prior_oof[va_idx] = np.bincount(y[tr_idx], minlength=3) / len(tr_idx)
results.append(summarise('Class frequencies', prior_oof))

num_base = [c for c in FEATURE_GROUPS['base'] if c not in CAT_COLS]
logreg = Pipeline([
    ('prep', ColumnTransformer([
        ('num', Pipeline([('impute', SimpleImputer(strategy='median', add_indicator=True)),
                          ('scale', StandardScaler())]), num_base),
        ('cat', Pipeline([('impute', SimpleImputer(strategy='constant', fill_value='missing')),
                          ('onehot', OneHotEncoder(handle_unknown='ignore'))]), CAT_COLS),
    ])),
    ('clf', LogisticRegression(max_iter=3000, C=1.0)),
])
to_obj = lambda X: X.astype({c: object for c in CAT_COLS})
logreg_oof = np.zeros((len(train), 3))
for tr_idx, va_idx in folds:
    logreg.fit(to_obj(train.iloc[tr_idx][FEATURE_GROUPS['base']]), y[tr_idx])
    logreg_oof[va_idx] = logreg.predict_proba(to_obj(train.iloc[va_idx][FEATURE_GROUPS['base']]))
results.append(summarise('Logistic regression', logreg_oof))

pd.DataFrame(results).set_index('model')

,OOF log loss,± fold std
model,,
Class frequencies,0.72352,0.00043
Logistic regression,0.46148,0.01235


## 4. Feature ablation

Feature groups are added one at a time with LightGBM, the fastest of the three boosters. A group is **kept only if it lowers log
loss by more than two standard errors of the paired per-fold difference**. Because every run uses the same folds, comparing fold by
fold removes most of the noise that comes from how the folds were split.

In [7]:
selected = list(FEATURE_GROUPS['base'])
current_oof, _ = run_cv('lightgbm', selected)
ablation = [{'step': 'base', 'OOF log loss': log_loss(y, current_oof), 'Δ vs current': np.nan, '2·SE': np.nan, 'kept': True}]

for group in ['mayo', 'ratios', 'adhoc']:
    candidate_oof, _ = run_cv('lightgbm', selected + FEATURE_GROUPS[group])
    diff = fold_losses(candidate_oof) - fold_losses(current_oof)  # negative = better
    two_se = 2 * diff.std(ddof=1) / np.sqrt(N_SPLITS)
    keep = diff.mean() < -two_se
    ablation.append({'step': f'+ {group}', 'OOF log loss': log_loss(y, candidate_oof),
                     'Δ vs current': diff.mean(), '2·SE': two_se, 'kept': keep})
    if keep:
        selected += FEATURE_GROUPS[group]
        current_oof = candidate_oof

print('Selected features:', len(selected))
pd.DataFrame(ablation).set_index('step')

Selected features: 18


,OOF log loss,Δ vs current,2·SE,kept
step,,,,
base,0.37591,NaN,NaN,True
+ mayo,0.37793,0.00202,0.00320,False
+ ratios,0.38149,0.00558,0.00206,False
+ adhoc,0.37717,0.00126,0.00172,False


### Check: class weighting

The setup above claims that up-weighting the rare CL class hurts log loss. Here's the measurement, using LightGBM on the selected
features.

In [8]:
balanced_oof, _ = run_cv('lightgbm', selected, class_weight='balanced')
print('unweighted        :', round(log_loss(y, current_oof), 5))
print("class_weight='balanced':", round(log_loss(y, balanced_oof), 5))
print('mean predicted P(CL): unweighted {:.3f} | balanced {:.3f} | true share {:.3f}'.format(
    current_oof[:, 1].mean(), balanced_oof[:, 1].mean(), (y == 1).mean()))

unweighted        : 0.37591
class_weight='balanced': 0.40051
mean predicted P(CL): unweighted 0.019 | balanced 0.037 | true share 0.025


## 5. Does the original dataset help?

Each booster is trained twice on the selected features: once on competition rows only, and once with the 418 real patients added to
every training fold, flagged with `is_original = 1`. Validation is always on competition rows only, so both runs are scored on the
same data.

In [9]:
runs = {}
settings = [False, True] if original is not None else [False]
for model_name in ['lightgbm', 'xgboost', 'catboost']:
    for use_original in settings:
        runs[(model_name, use_original)] = run_cv(model_name, selected, use_original)

rows = []
for (model_name, use_original), (oof, _) in runs.items():
    row = summarise(model_name, oof)
    row['training data'] = 'competition + original' if use_original else 'competition only'
    rows.append(row)
original_table = pd.DataFrame(rows).set_index(['model', 'training data'])
original_table

OOF log loss  ± fold std
model    training data                                   
lightgbm competition only             0.37591     0.00962
         competition + original       0.37292     0.00944
xgboost  competition only             0.37481     0.01045
         competition + original       0.37114     0.00971
catboost competition only             0.38539     0.00909
         competition + original       0.38273     0.00859

In [10]:
# Same paired-difference rule as the feature ablation: use the original data for a model only if it clearly helps.
use_original_for = {}
for model_name in ['lightgbm', 'xgboost', 'catboost']:
    if original is None:
        use_original_for[model_name] = False
        continue
    diff = fold_losses(runs[(model_name, True)][0]) - fold_losses(runs[(model_name, False)][0])
    two_se = 2 * diff.std(ddof=1) / np.sqrt(N_SPLITS)
    use_original_for[model_name] = diff.mean() < -two_se
    print(f'{model_name:9s} Δ = {diff.mean():+.5f}  (2·SE = {two_se:.5f})  -> use original: {use_original_for[model_name]}')

lightgbm  Δ = -0.00299  (2·SE = 0.00229)  -> use original: True
xgboost   Δ = -0.00366  (2·SE = 0.00110)  -> use original: True
catboost  Δ = -0.00266  (2·SE = 0.00108)  -> use original: True


## 6. Blend

Blend weights are fitted by **minimising OOF log loss**, constrained to be non-negative and sum to 1. With only three weights fitted
on 15,000 OOF rows, the risk of overfitting the blend is negligible.

In [11]:
model_names = ['lightgbm', 'xgboost', 'catboost']
oofs = [runs[(m, use_original_for[m])][0] for m in model_names]
tests = [runs[(m, use_original_for[m])][1] for m in model_names]

blend = lambda w, preds: sum(wi * p for wi, p in zip(w, preds))
opt = minimize(lambda w: log_loss(y, blend(w, oofs)), x0=np.full(3, 1 / 3), method='SLSQP',
               bounds=[(0, 1)] * 3, constraints={'type': 'eq', 'fun': lambda w: w.sum() - 1})
weights = opt.x
blend_oof, blend_test = blend(weights, oofs), blend(weights, tests)

for m, w in zip(model_names, weights):
    print(f'{m:9s} weight = {w:.3f}  (original data: {use_original_for[m]})')

lightgbm  weight = 0.280  (original data: True)
xgboost   weight = 0.672  (original data: True)
catboost  weight = 0.048  (original data: True)


In [12]:
final = results + [summarise(m, oof) for m, oof in zip(model_names, oofs)] + [summarise('Blend', blend_oof)]
results_table = pd.DataFrame(final).set_index('model')
results_table

,OOF log loss,± fold std
model,,
Class frequencies,0.72352,0.00043
Logistic regression,0.46148,0.01235
lightgbm,0.37292,0.00944
xgboost,0.37114,0.00971
catboost,0.38273,0.00859
Blend,0.37060,0.00946


## 7. Submission

In [13]:
submission = pd.DataFrame(blend_test, columns=[f'Status_{c}' for c in CLASSES])
submission.insert(0, 'id', test_raw['id'].to_numpy())
submission.iloc[:, 1:] = submission.iloc[:, 1:].div(submission.iloc[:, 1:].sum(axis=1), axis=0)

assert len(submission) == len(test_raw) and submission.notna().all().all()
assert list(submission.columns) == ['id', 'Status_C', 'Status_CL', 'Status_D']

submission.to_csv('submission.csv', index=False)
print('Saved submission.csv', submission.shape)
submission.head()

Saved submission.csv (10000, 4)


,id,Status_C,Status_CL,Status_D
0,15000,0.44094,0.00341,0.55564
1,15001,0.87779,0.01392,0.10829
2,15002,0.74841,0.00572,0.24588
3,15003,0.98084,0.00059,0.01857
4,15004,0.21789,0.10839,0.67372


## Findings

1. **Gradient boosting clearly beats the linear baseline.** Log loss drops from 0.461 (logistic regression) to 0.371 (blend). Predicting class frequencies alone scores 0.724.
2. **The original dataset helps.** Adding the 418 real patients to every training fold (just 3% more rows) lowers log loss by 0.003–0.004 for all three boosters, more than 2 SE each time.
3. **The Mayo risk score did *not* help.** It was the one domain-grounded feature, and it failed the ablation (+0.002, within noise). So did the ratio features and the ad-hoc scores. The boosters already rebuild these combinations from the raw lab values, so the final model uses only the raw features.
4. **Class weighting hurts under log loss.** `class_weight='balanced'` makes LightGBM worse, from 0.376 to 0.401, because it nearly doubles the predicted CL probability (0.037 vs a true share of 0.025).
5. **The blend adds little.** The optimised weights favour XGBoost (0.67), and the blend improves on the best single model by only 0.0005. The three boosters make very similar errors.

**Limitations:** hyperparameters are sensible defaults, not tuned; scores come from a single 5-fold split with one seed.